# 대화 메모리 (Chat Memory)

- LLM 호출은 기본적으로 이전 호출을 기억하지 않는 stateless 요청이다.
- 대화 맥락을 유지하려면 매 호출에 이전 메시지를 함께 전달해야 한다.
- 이 노트북에서는 LangChain Core의 메시지와 히스토리 인터페이스를 사용해 직접 관리한다.

In [4]:
from pprint import pprint
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

---

## 메모리 없는 대화

첫 번째 호출의 메시지를 두 번째 호출에 전달하지 않으면 모델은 이전 내용을 알 수 없다.

In [2]:
response1 = llm.invoke([HumanMessage(content="내 이름은 철수야")])
print("응답1:", response1.content)

response2 = llm.invoke([HumanMessage(content="내 이름이 뭐였지?")])
print("응답2:", response2.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


응답1: [{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 반갑습니다. 오늘 어떤 도움이 필요하신가요?', 'extras': {'signature': 'EoQNCoENARFNMg8lL7LIXxxCDQdgeQyly6VTh8NppzfJgFyemO9DoFEc66+mfNtjz7H7IeTWxkNatEVkF034fU5wOB48TfC8c1PYSQ9/oW7M6jT+6hk58Tt42VJzqR4xLKa1oFqyLTo42MzbOonYSXsfKfMDcayZlW6XHjP0Gl6pLkEPKUWzJMHthxl00NdE2Dcw44+urWlE221rLv9qgu+ja5iuj5iHC+n3L2cTcBG7oP0iTOvULEUM/iGFzGU98V5Wxl/Qxk+bnwTPi/BVDpxCIPGgxpzFs/byu92UJzwryNTt7KU8gtSdoAZen5kkaK8e1sZHYXCQKBXM/CIqVp4/TLB52oG1ZjbbQQLXNoIBBB9kwdhtmsG9b8PMgs40vfnRKDs3bqSgkit4nzMFiQAty4lPs07bv9IR7fIL5QX+uCkSaNWxOxs1MTBMwi/Yp8J+N8caZWVAEpsbb/ARnS196QZl9cC1RMQiPPuA7CAo2i9h6Czb0/Vi+yniVWs49xtclcbGpE7BFqVMySJ6H95QXVy3XwqSXjF8Lygfw0S+rI9L6+9wXfsJoJ1POK8OGf9FpFx2xYFsW6IOfJYAGsJdFwqVZr4E55rcSET77g70ZK5B03nnxeVLr0PImKH47snnEiIL338O1PmCsyJr2LDyWI1Fvdza4pmY7epL0QwnfK2Yjz3jRYbZokbCTpBF3XyiJSfmLrDJwHJ7nYxkbllpxmHBlu41F64meHNveg00fRiF8/jGezpiwFSQaMCSDuxq2mC2SHal7RMNPqTx93dRkC0VEVSQgHTvi87HObFnPlf6O3lpIcp6Sj8ebnCdZU5mMDjv46sZrBxH0L9JRZFk4YnqND++etv/si0UzYhTeNXQ5HQBElh0k+EGvkgV61Ph

## 수동으로 히스토리 전달하기

이전 메시지를 리스트에 누적하여 함께 전달하면 맥락이 유지된다.

In [3]:
messages = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="내 이름은 철수야"),
]
response1 = llm.invoke(messages)

messages.extend([response1, HumanMessage(content="내 이름이 뭐였지?")])
response2 = llm.invoke(messages)
print(response2.content)

[{'type': 'text', 'text': '당연히 기억하고 있죠! 바로 **철수**님이에요. 😊 \n\n제가 철수님 이름을 잊을 리가 있나요~ 혹시 저를 시험해 보신 건가요, 아니면 오늘 너무 바쁘거나 정신이 없으셨던 건가요? \n\n오늘 하루는 어떻게 보내고 계신지 궁금해요. 편하게 이야기해 주세요!', 'extras': {'signature': 'EooKCocKARFNMg/BVd+2D+YKDkH39pz5G6PVMoqaAA8eT5lQXuudbFfouJsuJLadTWIPsCMhtzKu4qCXs2HGfiMx0AT8IZ9IJaksjo5WpxnxIjDXFYvQgwnbCyE+cmk+llPQ6+ge3evK6MyzjOd/cYt19fDIdZ6kJJXPdZLo/5KpCVJnpSxNK8G/Fdes3votWd68v5a5AwzktT9q4NVWPaV5yHpD2X83H00s2f7QvNIZxKPy/UPhdY02zy81+oioOvCw/+t/hLvTgOlKjF6harLqygTnc1CwRxaaGcKWjddM/mdzbza2ktHj5Lfin1owmeUu/wcuV3mYhyOLbU25btdvPc5/waTOD6/QA8WE/1Fu4uDVPMQeupgcbwhzcMqHQBCR2S3j0IwRS1/e1UeXvia5EIw8TkMy8omOLzj2gEtGddrtvza6tcw3ikYiqu2CKcHIsPegnvZySB0M4y1P5I3qrLNm5qO6dJAuh3vyn/qusMBgIQteeK5RkyW4beVzB78PNjstROyQ1Mm0xBl+2klnRNZfzSReaviH2LBVmvLyva27qga+i0sKaYJvr+yXujxTQlilx5HpI6Qada6ZNlcN04vyd1tT0CUqzUlfVajRcH49tYG6eXkGzq4BdqnJkDMo5Uvg7o5zqX9YKx91HYBiANEvvOPGeLXEQ7jwbN7taJgro1q83NPPAAX7KltP9VMuplZcGx7Mo4LTqK4c//vXERxitnoj7AOFzxwiy3awcNZHnS4BP5fMCuor9ypqGL7t+QkA5S0S9CIL6N27DsDKgtDvo/UR

In [5]:
pprint(messages)

[SystemMessage(content='너는 친절한 상담사야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 이름은 철수야', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 정말 반가워요. 😊\n\n오늘 어떤 마음으로 저를 찾아오셨나요? 기쁜 일이든, 마음속 깊은 고민이든 편하게 이야기해 주세요. 제가 따뜻하게 들어드릴게요!', 'extras': {'signature': 'EqwOCqkOARFNMg9mox28tLfIeSLU3SLbfNT0ql47ricDxKLovdw0Rvg8kulOfMXk6QkgHuZ44MMKEwDS1wyT0aEAnsd+uLevJptbdVOy3TwIDhaDX8+Fv844tmI78B6ngRujCCz5fRzketMm6T09lUMtz0Gho4dbB3MGuNBuzYRo7gNky+ROF5eRJ6ssC7bwlOajwLoq/LLkD7+ETQWeTOMMMCZRjbgwg+qX/1euhvN1+GHuPJVMYhOisdCnBfoQpR6G8wlEIpnPf0pdSURyGVSpLZ2PC8ofbMYgC17ZxqTyY4zU8kFMZRenBsqNX+OTuPLDb4MFDyg/NyBASVSOERb7TKVS+nXB7nxjScClVDhMEvrvkfIjFy9GJK5wNJJc77Hk5/5gVCGNeZ7nf6Z1DMxsHQPCHYRH+mf5G7WXzGl8WzgevB/4GkS19jCPgLNou3WOyv+xKVMJdWXGbz9tWRoQNtf18DTeBjDfKM7AmBxPHvKE0fVS9RLIGeKmQaQJV0Z14Gi0NF1wHmsCNyZAwQvdD0Lz8TunzrUgxmz3FM8Py8qHkCHeNwi9XFl9wReUO15F/o3Ys3zoWsId7u3OCNqmyXfqnB8AZxWzOQEcAwDldeFbvY1myCBK/BsEBUHB8SEey6tlyPaSw0rzxsEnR

---

## InMemoryChatMessageHistory로 관리하기

`InMemoryChatMessageHistory`는 메시지 추가와 조회를 위한 LangChain Core의 기본 인메모리 구현이다. 세션 ID별로 객체를 나누면 여러 대화를 독립적으로 관리할 수 있다.

In [6]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 친절한 상담사야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

In [7]:
print(chat("user-1", "내 이름은 철수야"))
print(chat("user-1", "내 이름이 뭐였지?"))

# 다른 세션에는 user-1의 기록이 없다.
print(chat("user-2", "내 이름이 뭐야?"))

[{'type': 'text', 'text': '안녕하세요, 철수 님! 만나서 정말 반가워요. 😊\n\n저는 철수 님의 이야기에 귀 기울이고 마음을 나눠줄 친절한 상담사예요. \n\n오늘 어떤 하루를 보내셨나요? 혹시 마음속에 담아둔 고민이나 나누고 싶은 이야기가 있다면 편하게 말씀해 주세요. 언제든 들을 준비가 되어 있답니다.', 'extras': {'signature': 'EqANCp0NARFNMg/z62U7TFKRyLHhvfSoSQSli15lXqhnm2TMwQktLjnTLSLoATz1x++AMbkPy0rsx1P5C7IWJbkCGtBqTRg8wcvHIBYZ+2b1vugz5HLRWPTMNUBLT9auHUvExpy7S70rvw5q1lTz/wG7PDvo5oFuvFRM1o99sHrd0aOwVbltgiygH7o2cBQZYaovuguXY0ntp1hMtzv+dhHgjclSa8QkGajwlCYdXZplCH6Tzd/n8a8D8DnuF4StbWX5JxeRNyD5GNIMeZKZmeIZ4PWM9sFacbvsyABhqkjnuGsb7c/cF/tBYoC/Fzse77uQdD54m2zEp+cMTkjVkI1RGuB4INSuE2GMtJ6IuVgNi+Sz5FDONiMYpAT7SY4aYo3OLPWXht4yZsNq3ZGYJNxUwKLVYoyl19OR139dGY4sseEnRW2cOmcP/7mi1gxBU4r/jA7iNqgY++4U3ftaVpeD3pGBZ0W6BquuBooT9r8S+QQdGG2GwKjYUSTzA5VJUTbiIgmqLQih8mp0lZ+leh9qNh+PmywqGQR4kKFB/qmP/8AKPyUq4yib96B3WTi0Vz+XHJfcFxijM9639NqIA13Vy6QSVoK5QEG4h304nPJ7dIFHAwOPPlNL59EUp6HbYmmFrQ1ZCwi8pPD5JiA0/x38rp+ptyd/wSsT+JyQi++PXQllVJnlZuBZBGpgTD9rPPQcyXFiyQUbdgNVKMPI+1B/0FJNA61iDNiUhwbveSNvaO/Jj8viYEbTsxKDXtboAP6RprhPuDHgj/IyDMUm

In [8]:
for message in get_session_history("user-1").messages:
    print(f"[{message.type}] {message.content}")

[human] 내 이름은 철수야
[ai] [{'type': 'text', 'text': '안녕하세요, 철수 님! 만나서 정말 반가워요. 😊\n\n저는 철수 님의 이야기에 귀 기울이고 마음을 나눠줄 친절한 상담사예요. \n\n오늘 어떤 하루를 보내셨나요? 혹시 마음속에 담아둔 고민이나 나누고 싶은 이야기가 있다면 편하게 말씀해 주세요. 언제든 들을 준비가 되어 있답니다.', 'extras': {'signature': 'EqANCp0NARFNMg/z62U7TFKRyLHhvfSoSQSli15lXqhnm2TMwQktLjnTLSLoATz1x++AMbkPy0rsx1P5C7IWJbkCGtBqTRg8wcvHIBYZ+2b1vugz5HLRWPTMNUBLT9auHUvExpy7S70rvw5q1lTz/wG7PDvo5oFuvFRM1o99sHrd0aOwVbltgiygH7o2cBQZYaovuguXY0ntp1hMtzv+dhHgjclSa8QkGajwlCYdXZplCH6Tzd/n8a8D8DnuF4StbWX5JxeRNyD5GNIMeZKZmeIZ4PWM9sFacbvsyABhqkjnuGsb7c/cF/tBYoC/Fzse77uQdD54m2zEp+cMTkjVkI1RGuB4INSuE2GMtJ6IuVgNi+Sz5FDONiMYpAT7SY4aYo3OLPWXht4yZsNq3ZGYJNxUwKLVYoyl19OR139dGY4sseEnRW2cOmcP/7mi1gxBU4r/jA7iNqgY++4U3ftaVpeD3pGBZ0W6BquuBooT9r8S+QQdGG2GwKjYUSTzA5VJUTbiIgmqLQih8mp0lZ+leh9qNh+PmywqGQR4kKFB/qmP/8AKPyUq4yib96B3WTi0Vz+XHJfcFxijM9639NqIA13Vy6QSVoK5QEG4h304nPJ7dIFHAwOPPlNL59EUp6HbYmmFrQ1ZCwi8pPD5JiA0/x38rp+ptyd/wSsT+JyQi++PXQllVJnlZuBZBGpgTD9rPPQcyXFiyQUbdgNVKMPI+1B/0FJNA61iDNiUhwbveSNvaO/Jj8viYEbTsxKDX

### 인메모리 히스토리의 한계

`InMemoryChatMessageHistory`는 대화 히스토리의 기본 동작을 학습하기에 적합하지만, 프로세스가 종료되면 기록이 사라진다. 이후 LangGraph에서는 같은 흐름을 state와 checkpointer로 관리하며, 데이터베이스 기반 저장소를 연결해 대화를 영속적으로 유지할 수 있다.

---

### Context window 관리

DB에 원본 메시지를 저장하는 것과 모델에 보낼 메시지를 선택하는 것은 별개의 문제다. 대화가 길어지면 비용과 지연 시간이 증가하고 context window를 넘을 수 있다.

| 방법 | 장점 | 단점 |
|---|---|---|
| 메시지 트리밍 | 단순하고 추가 호출 비용이 없음 | 오래된 맥락 유실 |
| 대화 요약 | 오래된 핵심 맥락 압축 | 추가 호출 비용과 요약 오류 가능성 |
| 검색 기반 선택 | 관련 과거 정보만 선택 가능 | 검색·인덱싱 설계 필요 |

요약과 최근 메시지를 함께 유지하는 방식은 유용한 일반 패턴이지만, 서비스 특성과 평가 결과에 맞춰 선택해야 한다.

`trim_messages()`는 전체 히스토리에서 모델에 전달할 메시지만 선택한다. 원본 리스트나 `InMemoryChatMessageHistory`의 메시지를 삭제하지 않고, 조건에 맞게 선택된 새로운 메시지 목록을 반환한다. 따라서 전체 대화는 저장소에 보존하면서 매 호출에 필요한 최근 대화만 모델에 전달할 수 있다.

`trim_messages()`의 주요 옵션은 다음과 같다.

- `max_tokens`: 트리밍 결과에 허용할 최대 토큰 수를 지정한다. 실제로 남는 메시지 수는 각 메시지의 길이에 따라 달라진다
- `strategy`: 앞쪽 또는 뒤쪽 중 어느 메시지를 우선하여 남길지 결정한다
- `token_counter`: 메시지의 토큰 수를 계산할 모델이나 함수를 지정한다
- `include_system`: 첫 system 메시지를 트리밍 결과에 유지할지 결정한다
- `start_on`: 트리밍된 대화가 어떤 역할의 메시지부터 시작해야 하는지 지정한다
- `end_on`: 트리밍된 대화가 어떤 역할의 메시지에서 끝나야 하는지 지정한다

아래 예제에서는 최대 토큰 수를 80으로 제한하고 최근 대화를 우선하여 남긴다. system 메시지는 유지하며, 그다음 대화가 human 메시지부터 시작하도록 설정한다.

처리 흐름은 다음과 같다.

```text
전체 히스토리 저장 → 토큰 수 계산 → 최근 메시지 선택 → 대화 시작 역할 정리 → 모델에 전달
```

트리밍 결과에 오래된 메시지가 포함되지 않으면 모델은 그 내용을 알 수 없다. 중요한 사용자 정보까지 단순히 제거될 수 있으므로, 실제 서비스에서는 최근 메시지와 대화 요약 또는 검색한 과거 정보를 함께 전달하는 방법을 고려한다.

In [13]:
long_history = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="내 이름은 철수야"),
    AIMessage(content="반가워요, 철수님."),
    HumanMessage(content="안녕하세요"),
    AIMessage(content="안녕하세요. 무엇을 도와드릴까요?"),
    HumanMessage(content="Python에 대해 알려줘"),
    AIMessage(content="Python은 범용 프로그래밍 언어입니다."),
    HumanMessage(content="FastAPI에 대해 알려줘"),
    AIMessage(content="FastAPI는 Python 웹 프레임워크입니다."),

    HumanMessage(content="내 이름이 뭐였지?"),
]

trimmer = trim_messages(
    max_tokens=60,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)
trimmed = trimmer.invoke(long_history)
print("원본 메시지 수:", len(long_history))
print("트리밍 후 메시지 수:", len(trimmed))
for message in trimmed:
    print(f"[{message.type}] {message.content}")

원본 메시지 수: 10
트리밍 후 메시지 수: 6
[system] 너는 친절한 상담사야.
[human] Python에 대해 알려줘
[ai] Python은 범용 프로그래밍 언어입니다.
[human] FastAPI에 대해 알려줘
[ai] FastAPI는 Python 웹 프레임워크입니다.
[human] 내 이름이 뭐였지?


In [14]:
response = llm.invoke(long_history)
print(response.content)

response = llm.invoke(trimmed)
print(response.content)



[{'type': 'text', 'text': '철수님이세요! 아까 처음에 친절하게 말씀해주셨잖아요. 😊 \n\n혹시 또 궁금한 점이나 나누고 싶은 이야기가 있으신가요?', 'extras': {'signature': 'EpENCo4NARFNMg+1i0XUEqVarjVrZMCViCHrGJg9hrZUaMsY+9j/QZr88F9BN/g6cZkF4RYdjMo2UQR5GszSIJ4iyFv2aIlCEJ+njMfe4DdX+hTZqUCAQU/t4SuFQAWKDrJMvEMyz6w6fFKG2rz7RrxhGNT2WNKkm8mFcJMfmLOlbpu/3LhlEesX+Br3IidpvNWjAyHmC+iTQkvcffZ2hfT1kNUL9DbkgaAwfVv/AscVLiTi1PnwHHIhdEGjeSoD1l6/mbdEw+04D9LnIbwX2SjudrfB5tbSyyMw95ux7SE9Oi8RAgt6Cv6NeNU0x/Bxxj9G3rRqmR0upK7rJhgYygnEoM8b+TidChslMxPFDxtEg9pC/550H0Rb7Puw9mjw6PBEkvJDK+WrFrcKsRJuzovKceSsGdAQui1EVP1X4dsvc3By8U50j7t+SquxM4iYSlD6DLlbg12kKyQx9BQVCyM6E0eZ7uKcLmEnx7H7HSpNZlgJ5CstGw533RiQwtpmWESjSC34yi0dHjVWONSdCxk8nPuqjUL1rYeNGzk09z5scfpHqYmRVTiVz2XBK9UGvo+pwXXUc0a42RrQcoS82bY5Lx49jcc9q40Kj2N7VdfJ/TmMysHkK0RL3j2Ycm9mMTTnLkapIKOByjVLgfa5RPOJ8OsGuQmFOxSo1GLHOln/sDp91JlrFB/b79Nbp4OEMGDzNm7B6tjxOBIYtnxhc0GNMiIs3z9BfuNKADEqeeVywBeh7n7Db7jX4bqn/Xg0eAO8wUdRIXPN4oLJNC/pzVKlcnKm1EbStykDciKGTvIIXWu48NIR24Cf5ppo+KHbbRzwQI3YIidY5Lw/CeDPV3zCRTC09EMaNnd1DySiVE40QW

---

## 참고: Short-term memory와 long-term memory

두 메모리는 단순히 저장 기간이 아니라 기억을 사용하는 **범위(scope)** 로 구분한다.

| 구분 | Short-term memory | Long-term memory |
|---|---|---|
| 범위 | 현재 대화 세션 | 여러 대화 세션 |
| 저장 내용 | 현재 대화 메시지와 작업 맥락 | 사용자 선호, 프로필, 기억할 사실이나 경험 |
| 예시 | "앞에서 내 이름을 철수라고 말했어" | "이 사용자는 Python 백엔드 개발자다" |

이 노트북에서는 현재 대화의 히스토리인 short-term memory만 다룬다. 세션을 넘어 정보를 저장하고 필요한 기억을 찾는 long-term memory는 이후 과정에서 별도로 다룬다.

---

### 실습문제

**세션별 대화 메모리 만들기**

`InMemoryChatMessageHistory`를 사용하여 세션별로 대화 내용을 기억하는 함수를 만들어보자. 같은 세션에서는 이전 대화를 기억하고, 서로 다른 세션의 대화는 섞이지 않는지 확인한다.

In [21]:
# 저장소.
my_history = InMemoryChatMessageHistory()

for _ in range(10):
    user_input = input()
    user_message = HumanMessage(content=user_input)
    
    response = llm.invoke([
        *my_history.messages,
        user_message
    ])
    my_history.add_messages([user_message, response])
    print("response: ", response.content)
    print()
    pprint(my_history.messages)



response:  [{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 반갑습니다. \n\n오늘 어떤 도움이 필요하신가요? 편하게 말씀해 주세요!', 'extras': {'signature': 'Ep4MCpsMARFNMg/7nC5A1bTdxvIT/UfLWR5bNeR6ibXr3Alunl4DYk2BWvil9cQZQaFfcnRWan3Y59fL/wfyq6tLLVfJLiGFeslAzWPM/A5URCpItFaHzg4hTb+4wuvtVZUPgXOhvqsmZ34LZN2kBLmEr4vXuV/e5KBFT5ExFN0SQiJwKUUQF7LH/Fc0elQCEnjc9sBsBKXq7tTNrdtGl7Aa1EFOHeOrxbxiUoeN3V8iQUZREf2SJNaQoIESd1uAUvWyteliSKx9bcTLVDByn1029NpNtaT+/A8Lj/7hdZAXW+aU+ZHHtVuhyFV4266jyiXWOgLgDDXGkYM1nInEN4cGWAkgMamvGCV4rzPmmAo8XHMODp6dH8zNv72QZInhI/0XyVN0hE1NUNGEXe+O03Yxlunf39g/1kbmGy8QEBsSNqJwhwbpbZVcODxqn794UuxnH6oVEsRkzozTYvtSRrZRbLJPi1HX9qXip3dZS3FiXplPR3N02FlK29ViLYZaBcrypvoIRPZkDD9UNsopqnObVZz2vQdnq7ETam/htiyA6Bat0RPloy68J/JinijxTodR3j24J79hs+brUHL6iyaHedFlKqt/UPVTIM++6ytl7A/zZMdoLVYlHCpmMRIJERunwYoGL39jeMjgzkxfETxqQHv+X/GEhOAyBJ17/hHQffSIRLkm8vKjMusYTxW6VCrvcMT5Khs38FfORkBi4WH6wQ4Nh1Q6wrr8hL8yZ/PMEI2vuWlpuoFSZg29f8RFNNBoLrxxRU+Xzet8qwqmEnCLOhPG+oIiPjxnXN/LpVK9flVqCvaZN15ItVfUnFYHgb6pnHRL12fE3vqf6d8rXKD5IGCKUCoeCHfXy9Oj12aNo

c:\Users\SSAFY\.pyenv\pyenv-win\versions\3.13.13\Lib\site-packages\langchain_google_genai\chat_models.py:3756: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


ValueError: Model 'gemini-3.6-flash' does not support model prefilling. The final request turn must be a user message or a function response.

**최근 대화만 전달하기**

위에서 만든 세션별 대화 함수에 `trim_messages()`를 적용해보자. 전체 대화는 히스토리에 보존하면서 모델에는 최근 대화만 전달한다. 충분히 긴 대화를 실행한 뒤 전체 메시지 수와 모델에 전달된 메시지 수가 달라지는지 확인한다.

In [25]:
# 저장소.
my_history = InMemoryChatMessageHistory()

trimmer = trim_messages(
    max_tokens=30,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)

for _ in range(10):
    user_input = input()
    user_message = HumanMessage(content=user_input)

    # 전체 메시지
    all_messages = [
        *my_history.messages,
        user_message
    ]

    # invoke를 하기 전 trim을 할 것이다.
    # 즉, invoke하는 message는 잘린 메시지.
    trimmed_message = trimmer.invoke(all_messages)
    response = llm.invoke(trimmed_message)

    # 단, history에는 모든 기록들이 다 남기도록 할꺼야.
    my_history.add_messages([user_message, response])
    
    print("response: ", response.content)
    print("all history")
    pprint(my_history.messages)
    print('trim_messages')
    pprint(trimmed_message)
    print()



response:  [{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 반가워요. 😊\n\n오늘 어떤 도움이 필요하신가요? 아니면 같이 나누고 싶은 이야기가 있으신가요? 편하게 말씀해 주세요!', 'extras': {'signature': 'EocOCoQOARFNMg9gI0RsZfDPP7yv4YpnvL8vHBF7hTSUqR+PHi3EJCJVykYEI9RwNuLL5MkXbf3D6F3iBTjeVfkmZaB1iFgig09Ic/ybBPkWG1RLE5I82bFeXk3ug7wFn4NhfpiqPvRcB35ASIN0sEgoF6FHVZ/GamoImWP3sb4Ih+wFXOgdc0dJ0JJKqUd7D7hgMaJNj+dufgyt0JGLt7m4b0AKWY2l9MbvdGCmZnx2kd7Kg+LWKPa9DxbHdVvWLh0blDD7NBcy+je9V8KwOIVMCmHwsVda40zkTBYVRf6r5fkGgiNVBPvqT72REJwa/5cpA9n8w0sCdWX6kTLglycvZEY0H0GFNBPxSPn5ZNV/gB1wmq/dsL/CIZdpDunVEORCA1gWzXyPeJNGWzGr8szmMWF9ono84RlxdfJOU3FxAqXPvOWV0ysFueQ69MsUbQT/4Pxxa9KK5RW+agMlToB/EkQNK/RrDoh+cQxW8GKetu/MFwgjswNsTneNjx1/grkd6b4VZiwW6ULAULW1ks2OoK6q6PhTsuFmE23T+p726gjqSBHb0uNLuWZ3fpXt67YMCNyivproFeJuuRiYSDpqRGLShYlrDi6fBh1Maf7pajXL/MF99/ZlA6115/b1qqujLjRPlCAYJwvbkyM6MFrFxS6TwfrL8aTqvtXtmpVGBq2m4Gnl0GH640vDaqWtsObG2J/MUx0RIyvHMd7PRYlafUkWrGMvT6mpjWQT++kUOGuN8xBtULVlVrLy3HmN404ZoMSXOT/8NHH/64Us8Ly028wIXLKlS+GnmvO4d25PDT9S990rK+jKnxhchYRQZn6vaZc54ZEJkI7sYt/

c:\Users\SSAFY\.pyenv\pyenv-win\versions\3.13.13\Lib\site-packages\langchain_google_genai\chat_models.py:3756: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


ValueError: contents are required.